In [1]:
import os
import sys
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv()

# --- FIX LỖI WINDOWS NGAY TRONG CODE ---
# Giả sử bạn để winutils ở: C:\sneaker-recsys\hadoop\bin\winutils.exe
# Thì HADOOP_HOME phải trỏ về: C:\sneaker-recsys\hadoop
os.environ['HADOOP_HOME'] = "C:\\sneaker-recsys\\hadoop" 
sys.path.append("C:\\sneaker-recsys\\hadoop\\bin") 
# ---------------------------------------

aws_access_key = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
bucket_name = os.getenv("S3_BUCKET_NAME")

spark = SparkSession.builder \
    .appName("SilverCleaningDev") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.540") \
    .config("spark.hadoop.fs.s3a.access.key", aws_access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", aws_secret_key) \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

print("✅ Spark Session đã lên sóng!")

✅ Spark Session đã lên sóng!


In [2]:
from pyspark.sql.functions import (
    col, coalesce, lit, regexp_extract, lower, when, 
    trim, regexp_replace, split, count
)
from pyspark.sql.types import StructType, StructField, StringType, MapType
import pandas as pd
from IPython.display import display

# ==========================================
# 1. SETUP & UTILS (CẤU HÌNH & HÀM BỔ TRỢ)
# ==========================================

# A. Định nghĩa Schema để đọc file JSON lộn xộn
manual_schema = StructType([
    StructField("name", StringType(), True),
    StructField("price", StringType(), True),
    StructField("url", StringType(), True),
    StructField("image_url", StringType(), True),
    StructField("specs", MapType(StringType(), StringType()), True)
])

# B. Hàm suy luận Purpose (Mục đích sử dụng)
def infer_purpose(name_col, line_col):
    txt = lower(name_col)
    return when(
        (txt.contains("running")) | (txt.contains("marathon")) | (txt.contains("jogging")), "Running"
    ).when(
        (txt.contains("basketball")) | (txt.contains("jordan")) | (txt.contains("curry")) | (txt.contains("lebron")), "Basketball"
    ).when(
        (txt.contains("soccer")) | (txt.contains("football")) | (txt.contains("cleat")), "Soccer"
    ).when(
        (txt.contains("hiking")) | (txt.contains("trail")) | (txt.contains("trekking")), "Hiking"
    ).when(
        (txt.contains("training")) | (txt.contains("gym")) | (txt.contains("crossfit")), "Gym & Training"
    ).when(
        (txt.contains("skate")), "Skateboarding"
    ).otherwise(coalesce(col("specs.`Performance/Activity`"), lit("Casual")))

# C. Hàm chuẩn hóa Material (Gom nhóm vật liệu)
def normalize_material(col_name):
    # Lấy vật liệu chính (trước dấu phẩy) và chuyển về chữ thường
    m = lower(trim(split(col_name, "[,/]").getItem(0)))
    return when(
        (m.contains("synthetic")) | (m.contains("faux")) | (m.contains("polyurethane")) | (m.contains("pu")), "Synthetic"
    ).when(
        (m.contains("mesh")), "Mesh"
    ).when(
        (m.contains("canvas")), "Canvas"
    ).when(
        (m.contains("suede")), "Suede"
    ).when(
        (m.contains("leather")) | (m.contains("nubuck")) | (m.contains("patent")) | (m.contains("skin")), "Leather"
    ).when(
        (m.contains("textile")) | (m.contains("fabric")) | (m.contains("knit")) | (m.contains("cloth")), "Textile"
    ).when(
        (m.contains("rubber")), "Rubber"
    ).otherwise("Unknown")

# D. Hàm tách tiền tệ
def extract_currency(price_col):
    return when(price_col.contains("US"), "USD") \
        .when(price_col.contains("GBP"), "GBP") \
        .when(price_col.contains("EUR"), "EUR") \
        .when(price_col.contains("AU"), "AUD") \
        .otherwise("USD")

# ==========================================
# 2. READ & TRANSFORM (ĐỌC & BIẾN ĐỔI)
# ==========================================

# Đọc dữ liệu Bronze
bronze_path = f"s3a://{bucket_name}/bronze/ebay_raw/*/*/*.json"
print(f"📥 Reading data from: {bronze_path}")
df_bronze = spark.read.option("multiline", "true").schema(manual_schema).json(bronze_path)

# Áp dụng Transformation (Chưa lọc vội để xem hàng xấu)
df_transformed = df_bronze.select(
    col("name").alias("product_name"),
    
    # Brand
    coalesce(col("specs.Brand"), regexp_extract(col("name"), r"^(\w+)", 1)).alias("brand"),
    
    # Category: Xóa chữ 's (Men's -> Men)
    trim(regexp_replace(coalesce(col("specs.Department"), lit("Unisex")), "'s", "")).alias("category"),
    
    # Style
    coalesce(col("specs.Style"), col("specs.`Shoe Shaft Style`"), lit("Sneaker")).alias("style"),
    
    # Type & Model
    coalesce(col("specs.Type"), lit("Athletic")).alias("type"),
    coalesce(col("specs.Model"), lit("Unknown")).alias("model"),
    
    # Purpose
    infer_purpose(col("name"), coalesce(col("specs.`Product Line`"), lit(""))).alias("purpose"),
    
    # Color: Lấy màu đầu tiên
    trim(split(coalesce(col("specs.Color"), col("specs.Colour"), lit("Unknown")), "[,/]").getItem(0)).alias("color"),
    
    # Material: Chuẩn hóa về nhóm chính
    normalize_material(coalesce(col("specs.`Upper Material`"), col("specs.Material"), lit("Unknown"))).alias("material"),
    
    # Price
    extract_currency(col("price")).alias("currency"),
    regexp_extract(col("price"), r"(\d+\.?\d*)", 1).cast("float").alias("price"),
    
    col("image_url")
)

# ==========================================
# 3. VISUALIZE BEFORE FILTERING (XEM TRƯỚC KHI LỌC)
# ==========================================
print("\n" + "="*50)
print(f"🛑 BEFORE FILTERING (Total: {df_transformed.count()} rows)")
print("Dữ liệu đã biến đổi nhưng còn chứa RÁC (Unknown, Empty Box...)")
print("="*50)
display(df_transformed.limit(10).toPandas())

# ==========================================
# 4. FILTERING (LỌC RÁC)
# ==========================================

df_clean = df_transformed.filter(
    col("image_url").isNotNull() &                      # 1. Phải có ảnh
    (col("price") > 0) &                                # 2. Giá > 0
    (lower(col("purpose")) != "not specified") &        # 3. Purpose rõ ràng
    (~lower(col("product_name")).contains("empty box")) & # 4. Không phải hộp rỗng
    (col("model") != "Unknown") &                       # 5. Phải biết tên dòng giày (Model)
    (col("color") != "Unknown") &                       # 6. Phải biết màu
    (col("material") != "Unknown")                      # 7. Phải biết chất liệu
)

# ==========================================
# 5. VISUALIZE AFTER FILTERING (XEM SAU KHI LỌC)
# ==========================================
count_after = df_clean.count()
print("\n" + "="*50)
print(f"✅ AFTER FILTERING (Total: {count_after} rows)")
print("Dữ liệu tinh khiết (Clean Silver)")
print("="*50)
display(df_clean.limit(10).toPandas())

# Thống kê nhanh
print(f"\n📉 Đã loại bỏ: {df_transformed.count() - count_after} dòng rác.")
from pyspark.sql.functions import count, when, isnan

def check_nulls(df):
    df.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in df.columns]).show()

print("📊 Kiểm tra số lượng giá trị NULL:")
check_nulls(df_clean)
df_clean.groupBy("category").count().show()

📥 Reading data from: s3a://sneaker-db/bronze/ebay_raw/*/*/*.json

🛑 BEFORE FILTERING (Total: 14963 rows)
Dữ liệu đã biến đổi nhưng còn chứa RÁC (Unknown, Empty Box...)


,product_name,brand,category,style,type,model,purpose,color,material,currency,price,image_url
0,Adidas Duramo 10 Women's Running Shoe Blue Foo...,adidas,Women,Sneaker,Athletic,Adidas Duramo 10,Running,Blue,Mesh,USD,39.950001,https://i.ebayimg.com/images/g/0YEAAeSwQWtpYV6...
1,PUMA Carina Lift Lace Up Womens White Sneakers...,Puma,Women,Sneaker,Athletic,PUMA Carina Lift Lace Up,Lace Up,White,Synthetic,USD,44.950001,https://i.ebayimg.com/images/g/hGcAAeSw4tVpF0~...
2,DQ8397-500 Nike Air Jordan 1 Mid SE Purple Vel...,Nike,Women,Sneaker,Athletic,Air Jordan 1 Mid SE,Basketball,Purple,Synthetic,USD,109.989998,https://i.ebayimg.com/images/g/qFUAAOSwHGpn9YT...
3,"PUMA Vis2k Lace Up Womens Black, Pink, White S...",Puma,Women,Sneaker,Athletic,PUMA Vis2k Lace Up,"Lace Up, Sportstyle",Black,Mesh,USD,34.950001,https://i.ebayimg.com/images/g/zFcAAeSwi41pFmC...
4,Adidas Swift Run 23 Women's Running Shoes Athl...,adidas,Women,Sneaker,Athletic,adidas Swift Run,Running,White,Synthetic,USD,49.950001,https://i.ebayimg.com/images/g/8OUAAeSwEPNoP2m...
5,Adidas Originals Gazelle Women's Running Shoe ...,adidas,Women,Sneaker,Athletic,adidas Originals Gazelle,Running,Pink,Suede,USD,59.950001,https://i.ebayimg.com/images/g/e~8AAeSwgf9pNvZ...
6,Nike V2K Run Pink Foam Arctic Pink Womens Runn...,Nike,Women,Sneaker,Athletic,Nike V2K Run,Running,Pink,Mesh,USD,129.990005,https://i.ebayimg.com/images/g/AVYAAeSwdTVo2~S...
7,2 PAIRS SNORS Shoelaces ROUND + FLAT 60-240cm ...,2,Unisex,Sneaker,Athletic,Unknown,Casual,Unknown,Unknown,EUR,5.990000,https://i.ebayimg.com/images/g/TGAAAOSwagRjidX...
8,New Adidas Cream VL Court Base Shoes Footwear ...,adidas,Women,Sneaker,Athletic,adidas Vl Court,"casual, Walking",White,Leather,USD,39.970001,https://i.ebayimg.com/images/g/a18AAeSwkRFpGn-...
9,Women's Quick Dry Water Shoes Foot safety Bare...,NORTIV 8,Women,Water Shoes,Athletic,Unknown,"Beach, Water Sports",Unknown,Synthetic,USD,23.990000,https://i.ebayimg.com/images/g/210AAeSw4mtpaIh...



✅ AFTER FILTERING (Total: 6287 rows)
Dữ liệu tinh khiết (Clean Silver)


,product_name,brand,category,style,type,model,purpose,color,material,currency,price,image_url
0,Adidas Duramo 10 Women's Running Shoe Blue Foo...,adidas,Women,Sneaker,Athletic,Adidas Duramo 10,Running,Blue,Mesh,USD,39.950001,https://i.ebayimg.com/images/g/0YEAAeSwQWtpYV6...
1,PUMA Carina Lift Lace Up Womens White Sneakers...,Puma,Women,Sneaker,Athletic,PUMA Carina Lift Lace Up,Lace Up,White,Synthetic,USD,44.950001,https://i.ebayimg.com/images/g/hGcAAeSw4tVpF0~...
2,DQ8397-500 Nike Air Jordan 1 Mid SE Purple Vel...,Nike,Women,Sneaker,Athletic,Air Jordan 1 Mid SE,Basketball,Purple,Synthetic,USD,109.989998,https://i.ebayimg.com/images/g/qFUAAOSwHGpn9YT...
3,"PUMA Vis2k Lace Up Womens Black, Pink, White S...",Puma,Women,Sneaker,Athletic,PUMA Vis2k Lace Up,"Lace Up, Sportstyle",Black,Mesh,USD,34.950001,https://i.ebayimg.com/images/g/zFcAAeSwi41pFmC...
4,Adidas Swift Run 23 Women's Running Shoes Athl...,adidas,Women,Sneaker,Athletic,adidas Swift Run,Running,White,Synthetic,USD,49.950001,https://i.ebayimg.com/images/g/8OUAAeSwEPNoP2m...
5,Adidas Originals Gazelle Women's Running Shoe ...,adidas,Women,Sneaker,Athletic,adidas Originals Gazelle,Running,Pink,Suede,USD,59.950001,https://i.ebayimg.com/images/g/e~8AAeSwgf9pNvZ...
6,Nike V2K Run Pink Foam Arctic Pink Womens Runn...,Nike,Women,Sneaker,Athletic,Nike V2K Run,Running,Pink,Mesh,USD,129.990005,https://i.ebayimg.com/images/g/AVYAAeSwdTVo2~S...
7,New Adidas Cream VL Court Base Shoes Footwear ...,adidas,Women,Sneaker,Athletic,adidas Vl Court,"casual, Walking",White,Leather,USD,39.970001,https://i.ebayimg.com/images/g/a18AAeSwkRFpGn-...
8,Nike Shoes Running Air Zoom Pegasus 40 Black W...,Nike,Women,Sneaker,Athletic,Nike Air Zoom Pegasus 40,Running,Black,Mesh,USD,59.990002,https://i.ebayimg.com/images/g/9ywAAeSwR~VpE00...
9,adidas Originals VL Court Bold in Green and Bl...,adidas,Women,Sneaker,Trainer,adidas VL Court Bold,"Dance, Driving, Running & Jogging, School, Tra...",Green,Suede,GBP,100.000000,https://i.ebayimg.com/images/g/JqQAAeSwar5pHEc...



📉 Đã loại bỏ: 8676 dòng rác.
📊 Kiểm tra số lượng giá trị NULL:
+------------+-----+--------+-----+----+-----+-------+-----+--------+--------+-----+---------+
|product_name|brand|category|style|type|model|purpose|color|material|currency|price|image_url|
+------------+-----+--------+-----+----+-----+-------+-----+--------+--------+-----+---------+
|           0|    0|       0|    0|   0|    0|      0|    0|       0|       0|    0|        0|
+------------+-----+--------+-----+----+-----+-------+-----+--------+--------+-----+---------+

+--------------------+-----+
|            category|count|
+--------------------+-----+
|             Apparel|    4|
|         Unisex Kids|    2|
|              womens|    6|
|                 Men| 3056|
|                Mens|   50|
|                Boys|    8|
|              Womens|   64|
|               Women| 2621|
|       Unisex Adults|  434|
|               Teens|    9|
|              Ladies|   16|
|              Female|    1|
|Women, Ladies, Girls|   

In [3]:
from pyspark.sql.functions import (
    col, coalesce, lit, regexp_extract, lower, when, 
    trim, regexp_replace, split, count, desc
)
from pyspark.sql.types import StructType, StructField, StringType, MapType
import pandas as pd
from IPython.display import display

# Cấu hình hiển thị Pandas (để xem được nhiều dòng hơn)
pd.set_option('display.max_rows', 100)

# ==========================================
# 1. SETUP & UTILS
# ==========================================

# A. Schema
manual_schema = StructType([
    StructField("name", StringType(), True),
    StructField("price", StringType(), True),
    StructField("url", StringType(), True),
    StructField("image_url", StringType(), True),
    StructField("specs", MapType(StringType(), StringType()), True)
])

# B. Hàm suy luận Purpose (Đã cập nhật: Cắt dấu phẩy)
def infer_purpose(name_col, line_col):
    txt = lower(name_col)
    raw_purpose = when(
        (txt.contains("running")) | (txt.contains("marathon")) | (txt.contains("jogging")), "Running"
    ).when(
        (txt.contains("basketball")) | (txt.contains("jordan")) | (txt.contains("curry")) | (txt.contains("lebron")), "Basketball"
    ).when(
        (txt.contains("soccer")) | (txt.contains("football")) | (txt.contains("cleat")), "Soccer"
    ).when(
        (txt.contains("hiking")) | (txt.contains("trail")) | (txt.contains("trekking")), "Hiking"
    ).when(
        (txt.contains("training")) | (txt.contains("gym")) | (txt.contains("crossfit")), "Gym & Training"
    ).when(
        (txt.contains("skate")), "Skateboarding"
    ).otherwise(coalesce(col("specs.`Performance/Activity`"), lit("Casual")))
    
    # Chỉ lấy giá trị đầu tiên trước dấu phẩy
    return trim(split(raw_purpose, "[,]").getItem(0))

# C. Hàm chuẩn hóa Material
def normalize_material(col_name):
    m = lower(trim(split(col_name, "[,/]").getItem(0)))
    return when(
        (m.contains("synthetic")) | (m.contains("faux")) | (m.contains("polyurethane")) | (m.contains("pu")), "Synthetic"
    ).when(
        (m.contains("mesh")), "Mesh"
    ).when(
        (m.contains("canvas")), "Canvas"
    ).when(
        (m.contains("suede")), "Suede"
    ).when(
        (m.contains("leather")) | (m.contains("nubuck")) | (m.contains("patent")), "Leather"
    ).when(
        (m.contains("textile")) | (m.contains("fabric")) | (m.contains("knit")), "Textile"
    ).when(
        (m.contains("rubber")), "Rubber"
    ).otherwise("Unknown")

# D. Hàm tách tiền tệ
def extract_currency(price_col):
    return when(price_col.contains("US"), "USD") \
        .when(price_col.contains("GBP"), "GBP") \
        .when(price_col.contains("EUR"), "EUR") \
        .when(price_col.contains("AU"), "AUD") \
        .otherwise("USD")

# ==========================================
# 2. READ & TRANSFORM
# ==========================================
bronze_path = f"s3a://{bucket_name}/bronze/ebay_raw/*/*/*.json"
df_bronze = spark.read.option("multiline", "true").schema(manual_schema).json(bronze_path)

# Bước đệm: Xử lý sơ bộ Category để dễ chuẩn hóa
raw_category_col = trim(regexp_replace(coalesce(col("specs.Department"), lit("Unisex")), "'s", ""))

df_transformed = df_bronze.select(
    col("name").alias("product_name"),
    
    # Brand
    coalesce(col("specs.Brand"), regexp_extract(col("name"), r"^(\w+)", 1)).alias("brand"),
    
    # Category: Chuẩn hóa Mens -> Men, Womens -> Women
    when(raw_category_col == "Mens", "Men")
    .when(raw_category_col == "Womens", "Women")
    .otherwise(raw_category_col).alias("category"),
    
    # Style: Cắt lấy giá trị đầu tiên trước dấu phẩy
    trim(split(coalesce(col("specs.Style"), col("specs.`Shoe Shaft Style`"), lit("Sneaker")), "[,]").getItem(0)).alias("style"),
    
    # Type & Model
    coalesce(col("specs.Type"), lit("Athletic")).alias("type"),
    coalesce(col("specs.Model"), lit("Unknown")).alias("model"),
    
    # Purpose (Đã xử lý cắt dấu phẩy trong hàm infer_purpose)
    infer_purpose(col("name"), coalesce(col("specs.`Product Line`"), lit(""))).alias("purpose"),
    
    # Color: Cắt lấy màu đầu tiên
    trim(split(coalesce(col("specs.Color"), col("specs.Colour"), lit("Unknown")), "[,/]").getItem(0)).alias("color"),
    
    # Material
    normalize_material(coalesce(col("specs.`Upper Material`"), col("specs.Material"), lit("Unknown"))).alias("material"),
    
    # Price
    extract_currency(col("price")).alias("currency"),
    regexp_extract(col("price"), r"(\d+\.?\d*)", 1).cast("float").alias("price"),
    
    col("image_url")
)

# ==========================================
# 3. FILTERING & DEDUPLICATION (QUAN TRỌNG)
# ==========================================

# A. Lọc rác (Giá trị null, unknown, rác)
df_filtered = df_transformed.filter(
    col("image_url").isNotNull() & 
    (col("price") > 0) & 
    (lower(col("purpose")) != "not specified") &
    (~lower(col("product_name")).contains("empty box")) &
    (col("category") != "N/A") &  # <--- Bỏ N/A Category
    (col("model") != "Unknown") &
    (col("color") != "Unknown") &
    (col("material") != "Unknown")
)

# B. Xóa trùng lặp (Deduplication) - GIỮ LẠI 1 SẢN PHẨM DUY NHẤT
count_before_dedup = df_filtered.count()
df_final = df_filtered.dropDuplicates(["product_name"])
count_after_dedup = df_final.count()

print("="*60)
print(f"📉 Đã xóa trùng lặp: {count_before_dedup - count_after_dedup} sản phẩm.")
print(f"✅ TỔNG SỐ SẢN PHẨM CUỐI CÙNG (FINAL COUNT): {count_after_dedup}")
print("="*60)

# ==========================================
# 4. RE-CHECK STATISTICS (THỐNG KÊ LẠI)
# ==========================================

def check_stats(df):
    cols_to_check = ['category', 'purpose', 'style', 'currency', 'material']
    for c in cols_to_check:
        print(f"\n🔍 [ {c.upper()} ] Distribution:")
        # Lấy top 15 giá trị phổ biến nhất
        pdf = df.groupBy(c).count().orderBy(desc("count")).limit(15).toPandas()
        display(pdf)

check_stats(df_final)

print("\n--- 10 SẢN PHẨM MẪU SAU KHI CLEAN ---")
display(df_final.limit(10).toPandas())

📉 Đã xóa trùng lặp: 1473 sản phẩm.
✅ TỔNG SỐ SẢN PHẨM CUỐI CÙNG (FINAL COUNT): 4823

🔍 [ CATEGORY ] Distribution:


,category,count
0,Men,2366
1,Women,2075
2,Unisex Adults,333
3,Ladies,12
4,Teens,9
5,Boys,6
6,Unisex,4
7,Apparel,3
8,womens,3
9,woman,2



🔍 [ PURPOSE ] Distribution:


,purpose,count
0,Casual,2923
1,Running,387
2,Walking,317
3,Basketball,188
4,Lace Up,146
5,Gym & Training,125
6,CrossFit,98
7,Hiking,86
8,Running & Jogging,84
9,Skateboarding,78



🔍 [ STYLE ] Distribution:


...

,currency,count
0,USD,3117
1,AUD,990
2,GBP,695
3,EUR,21



🔍 [ MATERIAL ] Distribution:


,material,count
0,Synthetic,1851
1,Leather,1433
2,Suede,693
3,Mesh,409
4,Textile,253
5,Canvas,149
6,Rubber,35



--- 10 SẢN PHẨM MẪU SAU KHI CLEAN ---


,product_name,brand,category,style,type,model,purpose,color,material,currency,price,image_url
0,#15 WC chely Woman Size 9.5,WC Chely,Women,Bootie,Boot,WC chely,Casual,Black,Synthetic,USD,45.000000,https://i.ebayimg.com/images/g/iroAAOSwe5BmPa6...
1,$1250 Church's Consul Black Men’s Leather Cap ...,Church's,Men,Oxford,Dress,Church's Consul,Casual,Black,Leather,USD,128.000000,https://i.ebayimg.com/images/g/tmYAAeSw0pJo4ad...
2,$129 Crevo Atty Women's Boots Size 9,CREVO,Women,Bootie,Boot,Crevo Atty,Casual,Black,Suede,USD,40.990002,https://i.ebayimg.com/images/g/4UsAAOSwQ3pjdPQ...
3,$144 women’s 8 DR. MARTENS 1460 PASCAL VIRGINI...,Dr. Martens,Women,Bootie,Boot,Dr. Martens Pascal,Casual,Black Virginia,Leather,USD,99.000000,https://i.ebayimg.com/images/g/6j4AAeSwRYdpTsy...
4,$1450 Bottega Veneta Women Black Yellow Platfo...,Bottega Veneta,Women,Bootie,Boot,630297VBS50,Casual,Black,Leather,USD,336.390015,https://i.ebayimg.com/images/g/Xv0AAOSwO6Bmuho...
5,$265 Coach Pheobe Coconut Tan Cowboy Mid Calf ...,Coach,Women,Bootie,Boot,Pheobe,Casual,Coconut,Suede,USD,152.990005,https://i.ebayimg.com/images/g/HE0AAOSwNlVkxdw...
6,$798 Stuart Weitzman Women's Highland Black Su...,Stuart Weitzman,Women,Over-the-Knee,Boot,Stuart Weitzman Highland,Casual,Black,Suede,USD,239.990005,https://i.ebayimg.com/images/g/UYIAAOSwwnVcbcX...
7,(W) Adidas Samba OG Maroon Off White JR8844,adidas,Women,Sneaker,Athletic,adidas Samba,Baseball,Maroon,Leather,USD,93.000000,https://i.ebayimg.com/images/g/XJ4AAeSwtbtpfGK...
8,(size 4-11) adidas Samba Black,adidas,Men,Sneaker,Athletic,adidas Samba,Running & Jogging,Black,Leather,USD,64.989998,https://i.ebayimg.com/images/g/M5MAAeSwpj9pFY7...
9,(size 4-11) adidas Samba White,adidas,Men,Sneaker,Athletic,adidas Samba,Running & Jogging,White,Leather,USD,59.990002,https://i.ebayimg.com/images/g/CBgAAeSwEp1pdIF...
